# 🎬 TMDB Data — Load & Merge

Loads all 5 tables from `movies.db` into DataFrames, then merges them into one wide DataFrame for analysis.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("movies.db")

# Load all 5 tables
df_movies       = pd.read_sql("SELECT * FROM movies", conn)
df_genres       = pd.read_sql("SELECT * FROM genres", conn)
df_movie_genres = pd.read_sql("SELECT * FROM movie_genres", conn)
df_cast         = pd.read_sql("SELECT * FROM cast", conn)
df_directors    = pd.read_sql("SELECT * FROM directors", conn)

conn.close()

print(f"movies:        {df_movies.shape}")
print(f"genres:        {df_genres.shape}")
print(f"movie_genres:  {df_movie_genres.shape}")
print(f"cast:          {df_cast.shape}")
print(f"directors:     {df_directors.shape}")

movies:        (1604, 18)
genres:        (19, 2)
movie_genres:  (4456, 3)
cast:          (7937, 6)
directors:     (1602, 4)


In [13]:
df_directors

,id,movie_id,director_name,person_id
0,1,157336,Christopher Nolan,525
1,2,27205,Christopher Nolan,525
2,3,24428,Joss Whedon,12891
3,4,155,Christopher Nolan,525
4,5,19995,James Cameron,2710
...,...,...,...,...
1597,1598,13600,Gil Kenan,57193
1598,1599,929590,Alex Garland,2036
1599,1600,2288,Mike Nichols,5342
1600,1601,9779,Ken Kwapis,29009


In [12]:
df_cast

,id,movie_id,actor_name,character,billing_order,person_id
0,1,157336,Matthew McConaughey,Cooper,0,10297
1,2,157336,Anne Hathaway,Brand,1,1813
2,3,157336,Michael Caine,Professor Brand,2,3895
3,4,157336,Jessica Chastain,Murph,3,83002
4,5,157336,Casey Affleck,Tom,4,1893
...,...,...,...,...,...,...
7932,7933,1294203,Asha Banks,Noah,0,3499290
7933,7934,1294203,Matthew Broome,Nick,1,3912097
7934,7935,1294203,Eve Macklin,Ella,2,1092908
7935,7936,1294203,Ray Fearon,William,3,56650


In [11]:
df_movie_genres

,movie_id,genre_id,genre_name
0,157336,12,Adventure
1,157336,18,Drama
2,157336,878,Science Fiction
3,27205,28,Action
4,27205,878,Science Fiction
...,...,...,...
4451,9779,18,Drama
4452,9779,35,Comedy
4453,9779,10749,Romance
4454,1294203,18,Drama


In [10]:
df_genres

,genre_id,genre_name
0,12,Adventure
1,14,Fantasy
2,16,Animation
3,18,Drama
4,27,Horror
5,28,Action
6,35,Comedy
7,36,History
8,37,Western
9,53,Thriller


In [9]:
df_movies

,movie_id,title,original_title,release_date,release_year,release_month,runtime,budget,revenue,vote_average,vote_count,popularity,original_language,overview,tagline,is_franchise,collection_name,fetched_at
0,11,Star Wars,Star Wars,1977-05-25,1977,5,121,11000000.0,775398007.0,8.201,22047,20.4698,en,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",1,Star Wars Collection,2026-03-17T13:44:11.989517
1,12,Finding Nemo,Finding Nemo,2003-05-30,2003,5,100,94000000.0,940335536.0,7.817,20353,15.6778,en,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,1,Finding Nemo Collection,2026-03-17T13:44:20.404083
2,13,Forrest Gump,Forrest Gump,1994-06-23,1994,6,142,55000000.0,677387716.0,8.463,29378,25.0402,en,A man with a low IQ has accomplished great thi...,The world will never be the same once you've s...,0,None,2026-03-17T13:43:57.440834
3,14,American Beauty,American Beauty,1999-09-15,1999,9,122,15000000.0,356296601.0,8.000,12873,10.4014,en,"Lester Burnham, a depressed suburban father in...",... look closer,0,None,2026-03-17T13:45:28.071263
4,15,Citizen Kane,Citizen Kane,1941-04-17,1941,4,119,839727.0,23218000.0,7.978,5966,4.7630,en,Newspaper magnate Charles Foster Kane is taken...,Some called him a hero...others called him a h...,0,None,2026-03-17T13:49:51.272446
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1599,1408208,Exit 8,８番出口,2025-08-28,2025,8,95,0.0,39142423.0,6.465,231,13.1064,ja,A man becomes increasingly desperate when he r...,Turn back.,0,None,2026-03-17T13:55:09.539687
1600,1412113,Squid Game: Making Season 2,오징어 게임: 시즌2 제작 이야기,2025-01-02,2025,1,28,0.0,0.0,8.402,442,5.7189,ko,"From set designs to character arcs, get exclus...",,0,None,2026-03-17T13:51:54.668937
1601,1419406,The Shadow's Edge,The Shadow's Edge,2025-08-16,2025,8,142,0.0,174400000.0,7.210,497,78.6013,zh,Macau Police brings the tracking expert police...,He's training a new generation of law enforcer...,0,None,2026-03-17T13:54:25.797850
1602,1456349,It Was Just an Accident,Un simple accident,2025-10-01,2025,10,103,6000000.0,9602793.0,7.186,451,15.1428,fr,An unassuming mechanic is reminded of his time...,,0,None,2026-03-17T13:55:02.587624


In [3]:
# Aggregate genres into a comma-separated string per movie
genres_agg = (
    df_movie_genres
    .groupby("movie_id")["genre_name"]
    .agg(lambda x: ", ".join(sorted(x)))
    .reset_index()
    .rename(columns={"genre_name": "genres"})
)

genres_agg.head()

,movie_id,genres
0,11,"Action, Adventure, Science Fiction"
1,12,"Adventure, Animation, Family"
2,13,"Comedy, Drama, Romance"
3,14,Drama
4,15,"Drama, Mystery"


In [4]:
# Aggregate cast into a comma-separated string per movie (ordered by billing)
cast_agg = (
    df_cast
    .sort_values(["movie_id", "billing_order"])
    .groupby("movie_id")["actor_name"]
    .agg(lambda x: ", ".join(x))
    .reset_index()
    .rename(columns={"actor_name": "top_cast"})
)

cast_agg.head()

,movie_id,top_cast
0,11,"Mark Hamill, Harrison Ford, Carrie Fisher, Pet..."
1,12,"Albert Brooks, Ellen DeGeneres, Alexander Goul..."
2,13,"Tom Hanks, Robin Wright, Gary Sinise, Sally Fi..."
3,14,"Kevin Spacey, Annette Bening, Thora Birch, Wes..."
4,15,"Orson Welles, Joseph Cotten, Dorothy Comingore..."


In [5]:
# Prepare directors — just keep movie_id and director_name
directors_slim = df_directors[["movie_id", "director_name"]].copy()

directors_slim.head()

,movie_id,director_name
0,157336,Christopher Nolan
1,27205,Christopher Nolan
2,24428,Joss Whedon
3,155,Christopher Nolan
4,19995,James Cameron


In [6]:
# Merge everything into one wide DataFrame
df = (
    df_movies
    .merge(genres_agg,     on="movie_id", how="left")
    .merge(cast_agg,       on="movie_id", how="left")
    .merge(directors_slim, on="movie_id", how="left")
)

print(f"Final merged shape: {df.shape}")
df.head()

Final merged shape: (1604, 21)


,movie_id,title,original_title,release_date,release_year,release_month,runtime,budget,revenue,vote_average,...,popularity,original_language,overview,tagline,is_franchise,collection_name,fetched_at,genres,top_cast,director_name
0,11,Star Wars,Star Wars,1977-05-25,1977,5,121,11000000.0,775398007.0,8.201,...,20.4698,en,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",1,Star Wars Collection,2026-03-17T13:44:11.989517,"Action, Adventure, Science Fiction","Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas
1,12,Finding Nemo,Finding Nemo,2003-05-30,2003,5,100,94000000.0,940335536.0,7.817,...,15.6778,en,"Nemo, an adventurous young clownfish, is unexp...",There are 3.7 trillion fish in the ocean. They...,1,Finding Nemo Collection,2026-03-17T13:44:20.404083,"Adventure, Animation, Family","Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton
2,13,Forrest Gump,Forrest Gump,1994-06-23,1994,6,142,55000000.0,677387716.0,8.463,...,25.0402,en,A man with a low IQ has accomplished great thi...,The world will never be the same once you've s...,0,None,2026-03-17T13:43:57.440834,"Comedy, Drama, Romance","Tom Hanks, Robin Wright, Gary Sinise, Sally Fi...",Robert Zemeckis
3,14,American Beauty,American Beauty,1999-09-15,1999,9,122,15000000.0,356296601.0,8.000,...,10.4014,en,"Lester Burnham, a depressed suburban father in...",... look closer,0,None,2026-03-17T13:45:28.071263,Drama,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes
4,15,Citizen Kane,Citizen Kane,1941-04-17,1941,4,119,839727.0,23218000.0,7.978,...,4.7630,en,Newspaper magnate Charles Foster Kane is taken...,Some called him a hero...others called him a h...,0,None,2026-03-17T13:49:51.272446,"Drama, Mystery","Orson Welles, Joseph Cotten, Dorothy Comingore...",Orson Welles


In [7]:
# Quick sanity check
print("Columns:")
print(df.columns.tolist())
print()
print("Nulls in key merged columns:")
for col in ["genres", "top_cast", "director_name"]:
    nulls = df[col].isna().sum()
    print(f"  {col:<20} {nulls} nulls")

Columns:
['movie_id', 'title', 'original_title', 'release_date', 'release_year', 'release_month', 'runtime', 'budget', 'revenue', 'vote_average', 'vote_count', 'popularity', 'original_language', 'overview', 'tagline', 'is_franchise', 'collection_name', 'fetched_at', 'genres', 'top_cast', 'director_name']

Nulls in key merged columns:
  genres               0 nulls
  top_cast             5 nulls
  director_name        2 nulls


In [8]:
# Sample row to verify the merge looks right
df[["title", "release_year", "genres", "top_cast", "director_name", "budget", "revenue", "vote_average"]].head(10)

,title,release_year,genres,top_cast,director_name,budget,revenue,vote_average
0,Star Wars,1977,"Action, Adventure, Science Fiction","Mark Hamill, Harrison Ford, Carrie Fisher, Pet...",George Lucas,11000000.0,775398007.0,8.201
1,Finding Nemo,2003,"Adventure, Animation, Family","Albert Brooks, Ellen DeGeneres, Alexander Goul...",Andrew Stanton,94000000.0,940335536.0,7.817
2,Forrest Gump,1994,"Comedy, Drama, Romance","Tom Hanks, Robin Wright, Gary Sinise, Sally Fi...",Robert Zemeckis,55000000.0,677387716.0,8.463
3,American Beauty,1999,Drama,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,15000000.0,356296601.0,8.000
4,Citizen Kane,1941,"Drama, Mystery","Orson Welles, Joseph Cotten, Dorothy Comingore...",Orson Welles,839727.0,23218000.0,7.978
5,The Fifth Element,1997,"Action, Adventure, Science Fiction","Bruce Willis, Milla Jovovich, Gary Oldman, Ian...",Luc Besson,90000000.0,263920180.0,7.564
6,Metropolis,1927,"Drama, Science Fiction","Gustav Fröhlich, Brigitte Helm, Alfred Abel, R...",Fritz Lang,5300000.0,1350322.0,8.100
7,Pirates of the Caribbean: The Curse of the Bla...,2003,"Action, Adventure, Fantasy","Johnny Depp, Geoffrey Rush, Orlando Bloom, Kei...",Gore Verbinski,140000000.0,655011224.0,7.820
8,Kill Bill: Vol. 1,2003,"Action, Crime","Uma Thurman, Lucy Liu, Vivica A. Fox, Daryl Ha...",Quentin Tarantino,30000000.0,180906076.0,7.971
9,Apocalypse Now,1979,"Drama, War","Martin Sheen, Marlon Brando, Frederic Forrest,...",Francis Ford Coppola,31500000.0,150000000.0,8.268
